[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/ddpm/ddpm.ipynb)

In [ ]:
import os
import sys
import subprocess

# Default Colab GPU preference: A100 (fallback: L4)
if "google.colab" in sys.modules and not os.path.exists("benchmark.py"):
    if not os.path.exists("vision_transformers"):
        subprocess.run(
            ["git", "clone", "https://github.com/a-ramakrishnan/vision_transformers.git"],
            check=True,
        )
    os.chdir("vision_transformers")
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "feat/larger-datasets-and-benchmarks"], check=True)
    os.chdir("ddpm")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
import json
import torch
from dataset import get_ddpm_dataloaders
from scheduler import LinearNoiseScheduler
from unet import UNet
from train import train_ddpm
from sample import plot_forward_diffusion, sample_ddpm, plot_ddpm_samples
from benchmark import benchmark_ddpm, save_benchmark_results

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f"Using device: {device} ({gpu_name})")

DATASETS_TO_RUN = ["cifar10", "eurosat"]
BATCH_SIZE = 128
NUM_TIMESTEPS = 1000
NUM_EPOCHS = 20
LR = 2e-4

In [ ]:
scheduler = LinearNoiseScheduler(num_timesteps=NUM_TIMESTEPS, beta_start=1e-4, beta_end=0.02, device=device)
all_training_results = {"runtime_gpu": gpu_name}
all_benchmark_results = {"runtime_gpu": gpu_name}

for ds_name in DATASETS_TO_RUN:
    print(f"\n==================== DATASET: {ds_name.upper()} ====================")
    train_loader, test_loader, cfg = get_ddpm_dataloaders(dataset_name=ds_name, batch_size=BATCH_SIZE)
    in_ch, img_sz = cfg["in_channels"], cfg["image_size"]

    plot_forward_diffusion(
        scheduler, train_loader, timesteps_to_show=[0, 100, 300, 600, 999],
        device=device, save_path=f"forward_diffusion_{ds_name}.png",
    )

    model = UNet(im_channels=in_ch).to(device)
    model, ddpm_hist = train_ddpm(
        model, scheduler, train_loader, num_epochs=NUM_EPOCHS, lr=LR, device=device,
        checkpoint_path=f"ddpm_{ds_name}.pth",
    )

    generated_imgs = sample_ddpm(
        model, scheduler, num_samples=16, im_size=img_sz, im_channels=in_ch, device=device
    )
    plot_ddpm_samples(
        generated_imgs, save_path=f"ddpm_{ds_name}_samples.png",
        title=f"{ds_name.upper()} DDPM: Generated Samples from Pure Gaussian Noise",
    )

    ddpm_bench = benchmark_ddpm(
        model, scheduler, test_loader, im_size=img_sz, im_channels=in_ch,
        device=device, history=ddpm_hist, num_gen_samples=128, batch_size=64, t_noise=250,
    )

    all_training_results[ds_name] = ddpm_hist
    all_benchmark_results[ds_name] = ddpm_bench
    print(f"\n[{ds_name.upper()}] DDPM Benchmark:", json.dumps(ddpm_bench, indent=2))

with open("training_results.json", "w") as f:
    json.dump(all_training_results, f, indent=2)
save_benchmark_results(all_benchmark_results, save_path="benchmark_results.json")